# Autoship Delay in Onboarding to Household (Adult + Kids) — Power Analysis

**Experiment:** Autoship Delay in Onboarding to Household (Adults + Kids) · **Owner:** Sergio Oyola · **Primary metric analyzed here:** Autoship Adoption Rate · **Randomization unit:** `client_id` · **Allocation point:** at Household onboarding, once eligibility criteria are met (before a Fix is scheduled)

This notebook sizes a 2-cell A/B test for its primary metric, **Autoship Adoption Rate**, using a historical, pre-launch read of the eligible population — no live allocation log exists yet, since this experiment has not launched.

## Population
Household onboarding clients that:
- Are not enrolled in Autoship
- Business Line: Women, Men, or Kids
- Are linked to a primary household client (`household_primary_client_id IS NOT NULL`) — no field marking the specific onboarding flow (e.g. a signup-type value) could be found in the warehouse, so this broader linkage condition is used instead and likely captures more clients than the exact intended population
- Client does **not** already have a Fix scheduled
- The primary client on the household already has shipping and payment info saved

These clients currently skip Look-Feel-Fit (LFF) entirely and see no Autoship nudge at any point during onboarding — the baseline measured below is their **current, un-nudged** adoption rate, i.e., what the Control experience looks like today.

## Design: 2-cell test, single comparison
Eligible clients are randomized at Household onboarding into 2 cells at a 50/50 split:

| Cell | Experience |
|---|---|
| Control | Status quo — no Autoship nudge at any point |
| Treatment | Routed into the existing Autoship Delay experience immediately after First Fix conversion |

A single pairwise comparison is planned (Treatment vs. Control), so `alpha = 0.05` needs no multiple-comparison adjustment. First Fix Conversion is tracked as a guardrail metric but is not powered here — this notebook sizes the primary metric only; guardrail risk is instead monitored qualitatively, with a directional decline triggering an early stop rather than a statistical test.

## Sidedness: both alternatives evaluated
The hypothesis is directional (Autoship Adoption Rate increases without decreasing First Fix conversion), which argues for a one-sided test — but a two-sided view is also worth having on hand, as the more conservative standard. Both are computed side by side in Step 2; the headline summary in Step 3 uses the one-sided view, matching the directional hypothesis.

## Maturation window
A client's Autoship-adoption signal needs time to resolve after onboarding — they enroll, decline, or abandon at some point afterward, not instantly. Step 0b checks this empirically for this population, rather than assuming a fixed number of days.

In [1]:
import numpy as np
import pandas as pd
from amphibian import get_data_accessor
from power import n_total_statsmodels

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql)

# Data parameters
BUSINESS_LINES = ('Womens', 'Mens', 'Kids')
COHORT_START = '2026-03-01'  # 6 months of pooled history is enough to size this population
MATURATION_DAYS = 90  # days to wait for a client's Autoship demand event to resolve after onboarding
VALIDATION_WINDOW_DAYS = 30  # short recent window used only for the Step 0 schema/logic check

# Design parameters (2-cell test, single comparison, no multiple-comparison correction)
ALPHA = 0.05
POWER = 0.80
N_ARMS = 2
SPLIT = 0.5  # Control and Treatment are equal-sized arms
MDE_GRID = [0.02, 0.03, 0.04, 0.05, 0.10]  # relative lift scenarios on Autoship Adoption Rate

## Step 0 — Confirm the eligibility fields resolve as expected

Eligibility is defined by four conditions pulled from three different tables, joined here for the first time:

- The "no Fix scheduled" check — no `curated.client_first_conversion` record at or before onboarding
- The primary client's "shipping and payment info saved" — read from `client_service_production.clients.shipping_address` (shipping) and a matching row in `payment_method_service.payment_methods` (payment on file)

`household_primary_client_id IS NOT NULL` is a confirmed, real column and is used directly as a filter here, so this check only scans household-linked clients — and only over the last `VALIDATION_WINDOW_DAYS` days, since this is a quick schema/logic check, not the baseline measurement itself. No signup-flow-type field exists anywhere in the warehouse to narrow this further, so `household_primary_client_id` alone stands in as the household filter and likely captures a broader population than intended.

In [2]:
validation_query = f"""--sql
WITH household_signups AS (
    SELECT
        c.client_id,
        c.household_primary_client_id,
        c.signup_at
    FROM curated.client c
    WHERE c.household_primary_client_id IS NOT NULL
      AND c.business_line IN {BUSINESS_LINES}
      AND c.signup_at >= CURRENT_DATE - INTERVAL '{VALIDATION_WINDOW_DAYS}' DAY
      AND COALESCE(c.fake_client_flag, 0) = 0
      AND COALESCE(c.employee_affiliated_flag, 0) = 0
),
flags AS (
    SELECT
        hs.client_id,
        (cfc.client_id IS NULL) AS no_fix_scheduled,
        (csp.client_id IS NOT NULL) AS primary_profile_found,
        (csp.shipping_address IS NOT NULL) AS shipping_saved,
        EXISTS (
            SELECT 1 FROM payment_method_service.payment_methods pm
            WHERE CAST(pm.client_id AS INTEGER) = hs.household_primary_client_id
        ) AS payment_saved
    FROM household_signups hs
    LEFT JOIN curated.client_first_conversion cfc
        ON cfc.client_id = hs.client_id
       AND cfc.cancellation_adjusted_first_fix_demand_ts <= hs.signup_at
    LEFT JOIN client_service_production.clients csp
        ON csp.client_id = hs.household_primary_client_id
)
SELECT
    COUNT(*) AS n_clients,
    COUNT(*) FILTER (WHERE no_fix_scheduled) AS n_without_fix_scheduled,
    COUNT(*) FILTER (WHERE primary_profile_found) AS n_primary_profile_found,
    COUNT(*) FILTER (WHERE shipping_saved) AS n_shipping_saved,
    COUNT(*) FILTER (WHERE payment_saved) AS n_payment_saved
FROM flags
"""

validation_df = query(validation_query)
validation_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_clients,n_without_fix_scheduled,n_primary_profile_found,n_shipping_saved,n_payment_saved
0,28011,28011,28011,28011,23876


**Reading this:** of 28,011 household-linked clients in the last 30 days, `n_without_fix_scheduled` and `n_primary_profile_found` both equal `n_clients` exactly (100%) — the joins resolve cleanly. `n_shipping_saved` is also 100%, and `n_payment_saved` is 23,876 (85.2%) — a large majority, not all, which is the expected shape (some primary clients haven't saved a payment method yet).

## Step 0b — How long adoption actually takes to resolve

The 90-day maturation buffer used in Step 1 is a starting assumption, not a derived one. This checks it directly: for a cohort old enough that a full year has already passed, what share of eventual adopters (adopting within 365 days of onboarding) had already adopted by day 7, 14, 30, 60, 90, 120, and 180? Wherever this curve flattens out is the maturation window that should actually be used.

In [3]:
maturation_check_query = f"""--sql
WITH household_cohort AS (
    SELECT
        c.client_id,
        c.household_primary_client_id,
        c.signup_at AS onboarding_ts
    FROM curated.client c
    WHERE c.business_line IN {BUSINESS_LINES}
      AND c.household_primary_client_id IS NOT NULL
      AND COALESCE(c.fake_client_flag, 0) = 0
      AND COALESCE(c.employee_affiliated_flag, 0) = 0
      AND c.signup_at <= CURRENT_DATE - INTERVAL '400' DAY
),
not_fix_scheduled AS (
    SELECT hc.*
    FROM household_cohort hc
    LEFT JOIN curated.client_first_conversion cfc
        ON cfc.client_id = hc.client_id
       AND cfc.cancellation_adjusted_first_fix_demand_ts <= hc.onboarding_ts
    WHERE cfc.client_id IS NULL
),
primary_profile_ready AS (
    SELECT nfs.*
    FROM not_fix_scheduled nfs
    JOIN client_service_production.clients csp
        ON csp.client_id = nfs.household_primary_client_id
    WHERE csp.shipping_address IS NOT NULL
      AND EXISTS (
          SELECT 1 FROM payment_method_service.payment_methods pm
          WHERE CAST(pm.client_id AS INTEGER) = nfs.household_primary_client_id
      )
),
not_yet_autoship AS (
    SELECT ppr.*
    FROM primary_profile_ready ppr
    LEFT JOIN curated.client_pulse_journal pj
        ON pj.client_id = ppr.client_id
       AND pj.start_date <= DATE(ppr.onboarding_ts)
       AND pj.end_date > DATE(ppr.onboarding_ts)
       AND pj.last_autoship_demand_ts IS NOT NULL
    WHERE pj.client_id IS NULL
),
fresh_demand AS (
    SELECT
        nya.client_id,
        nya.onboarding_ts,
        MIN(pj.last_autoship_demand_ts) AS first_post_onboarding_demand_ts
    FROM not_yet_autoship nya
    JOIN curated.client_pulse_journal pj
        ON pj.client_id = nya.client_id
       AND pj.last_autoship_demand_ts > nya.onboarding_ts
    GROUP BY nya.client_id, nya.onboarding_ts
),
days_to_adopt AS (
    SELECT
        nya.client_id,
        DATE_DIFF('day', nya.onboarding_ts, fd.first_post_onboarding_demand_ts) AS days_to_adopt
    FROM not_yet_autoship nya
    LEFT JOIN fresh_demand fd ON fd.client_id = nya.client_id
)
SELECT
    t.checkpoint_days,
    COUNT(*) FILTER (WHERE days_to_adopt <= t.checkpoint_days) AS n_adopted_by_checkpoint,
    COUNT(*) FILTER (WHERE days_to_adopt <= 365) AS n_adopted_within_365d,
    COUNT(*) FILTER (WHERE days_to_adopt <= t.checkpoint_days) * 1.0
        / NULLIF(COUNT(*) FILTER (WHERE days_to_adopt <= 365), 0) AS share_of_eventual_adopters_captured
FROM days_to_adopt
CROSS JOIN UNNEST(ARRAY[7, 14, 30, 60, 90, 120, 180]) AS t(checkpoint_days)
GROUP BY t.checkpoint_days
ORDER BY t.checkpoint_days
"""

maturation_check_df = query(maturation_check_query)
maturation_check_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,checkpoint_days,n_adopted_by_checkpoint,n_adopted_within_365d,share_of_eventual_adopters_captured
0,7,265287,656652,0.4
1,14,299130,656652,0.5
2,30,342213,656652,0.5
3,60,394235,656652,0.6
4,90,436456,656652,0.7
5,120,472682,656652,0.7
6,180,529309,656652,0.8


**Reading this:** the observed curve is 40% (day 7) → 46% (14) → 52% (30) → 60% (60) → 66% (90) → 72% (120) → 81% (180) of eventual (365-day) adopters captured — still climbing meaningfully at day 180, not flattened. This is the opposite of "90 days is too long": at 90 days, roughly a third of eventual adopters haven't shown up yet, so `MATURATION_DAYS = 90` in Step 1 likely **understates** the true adoption rate. The checkpoint grid would need to extend past 180 days to find where the curve actually levels off before picking a final number with confidence.

## Step 1 — Autoship Adoption Rate baseline & daily eligible volume

Cohort = clients meeting all four eligibility conditions, pooled over the 6 months from `COHORT_START` through a 90-day-mature cutoff — recent enough to reflect the current population without scanning more history than sizing needs. Adoption is read from `curated.client_pulse_journal.last_autoship_demand_ts`, scanning each client's **full** journal history for the earliest demand timestamp *after* onboarding, since a client's demand timestamp resets to null on a full cancellation and a snapshot-only read would under-count adoption. Because these clients see no nudge today, this is a read of the **current, un-nudged** adoption rate — i.e., what Control looks like.

In [4]:
baseline_query = f"""--sql
WITH household_cohort AS (
    SELECT
        c.client_id,
        c.household_primary_client_id,
        c.signup_at AS onboarding_ts
    FROM curated.client c
    WHERE c.business_line IN {BUSINESS_LINES}
      AND c.household_primary_client_id IS NOT NULL
      AND COALESCE(c.fake_client_flag, 0) = 0
      AND COALESCE(c.employee_affiliated_flag, 0) = 0
      AND c.signup_at >= DATE '{COHORT_START}'
      AND c.signup_at <= CURRENT_DATE - INTERVAL '{MATURATION_DAYS}' DAY
),
not_fix_scheduled AS (
    SELECT hc.*
    FROM household_cohort hc
    LEFT JOIN curated.client_first_conversion cfc
        ON cfc.client_id = hc.client_id
       AND cfc.cancellation_adjusted_first_fix_demand_ts <= hc.onboarding_ts
    WHERE cfc.client_id IS NULL
),
primary_profile_ready AS (
    SELECT nfs.*
    FROM not_fix_scheduled nfs
    JOIN client_service_production.clients csp
        ON csp.client_id = nfs.household_primary_client_id
    WHERE csp.shipping_address IS NOT NULL
      AND EXISTS (
          SELECT 1 FROM payment_method_service.payment_methods pm
          WHERE CAST(pm.client_id AS INTEGER) = nfs.household_primary_client_id
      )
),
not_yet_autoship AS (
    SELECT ppr.*
    FROM primary_profile_ready ppr
    LEFT JOIN curated.client_pulse_journal pj
        ON pj.client_id = ppr.client_id
       AND pj.end_date > DATE '{COHORT_START}'
       AND pj.start_date <= DATE(ppr.onboarding_ts)
       AND pj.end_date > DATE(ppr.onboarding_ts)
       AND pj.last_autoship_demand_ts IS NOT NULL
    WHERE pj.client_id IS NULL
),
fresh_demand AS (
    SELECT
        nya.client_id,
        MIN(pj.last_autoship_demand_ts) AS first_post_onboarding_demand_ts
    FROM not_yet_autoship nya
    JOIN curated.client_pulse_journal pj
        ON pj.client_id = nya.client_id
       AND pj.last_autoship_demand_ts > DATE '{COHORT_START}'
       AND pj.last_autoship_demand_ts > nya.onboarding_ts
    GROUP BY nya.client_id
),
joined AS (
    SELECT
        nya.client_id,
        DATE_TRUNC('month', nya.onboarding_ts) AS month,
        DATE(nya.onboarding_ts) AS onboarding_date,
        CASE WHEN fd.first_post_onboarding_demand_ts IS NOT NULL
              AND fd.first_post_onboarding_demand_ts <= nya.onboarding_ts + INTERVAL '{MATURATION_DAYS}' DAY
             THEN 1 ELSE 0 END AS adopted_autoship
    FROM not_yet_autoship nya
    LEFT JOIN fresh_demand fd ON fd.client_id = nya.client_id
),
month_days AS (
    SELECT month, COUNT(DISTINCT onboarding_date) AS days_observed
    FROM joined
    GROUP BY 1
)
SELECT
    j.month,
    COUNT(*) AS n_eligible,
    AVG(CAST(j.adopted_autoship AS DOUBLE)) AS autoship_adoption_rate,
    md.days_observed,
    COUNT(*) * 1.0 / md.days_observed AS eligible_per_day
FROM joined j
JOIN month_days md ON md.month = j.month
GROUP BY j.month, md.days_observed
ORDER BY j.month DESC
"""

baseline_df = query(baseline_query)
baseline_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,n_eligible,autoship_adoption_rate,days_observed,eligible_per_day
0,2026-06-01 00:00:00.000,4674,0.102054,9,519.3
1,2026-05-01 00:00:00.000,17284,0.093497,31,557.5
2,2026-04-01 00:00:00.000,17734,0.094733,30,591.1
3,2026-03-01 00:00:00.000,21256,0.110745,31,685.7


**Reference month:** pick the most recent calendar month whose onboarding dates are all at least 90 days old as of today — that's **May 2026** (June 2026 shows only 9 of its days past the cutoff, so it's context only, not the baseline). Monthly rates for context: March 11.1%, April 9.5%, May 9.3%.

In [5]:
REFERENCE_MONTH = '2026-05-01'
ref = baseline_df[baseline_df['month'].astype(str).str.startswith(REFERENCE_MONTH)].reset_index(drop=True)

BASELINE_RATE = float(ref['autoship_adoption_rate'][0])
DAILY_ELIGIBLE = float(ref['eligible_per_day'][0])

print(f"BASELINE_RATE (un-nudged, current Control experience) = {BASELINE_RATE:.4f}  |  DAILY_ELIGIBLE (reference month) = {DAILY_ELIGIBLE:,.1f} clients/day")

BASELINE_RATE (un-nudged, current Control experience) = 0.0935  |  DAILY_ELIGIBLE (reference month) = 557.5 clients/day


## Step 1b — A fresher, decoupled daily-volume read

Unlike the adoption rate, daily eligible volume doesn't need the 90-day maturation wait — all four eligibility conditions are known immediately at onboarding. The most recent complete calendar month is used here for volume, decoupled from the matured reference month used for the rate above, in case the two differ.

In [6]:
recent_volume_query = f"""--sql
WITH household_cohort AS (
    SELECT c.client_id, c.household_primary_client_id, c.signup_at AS onboarding_ts
    FROM curated.client c
    WHERE c.business_line IN {BUSINESS_LINES}
      AND c.household_primary_client_id IS NOT NULL
      AND COALESCE(c.fake_client_flag, 0) = 0
      AND COALESCE(c.employee_affiliated_flag, 0) = 0
      AND c.signup_at >= DATE_TRUNC('month', CURRENT_DATE) - INTERVAL '1' MONTH
      AND c.signup_at < DATE_TRUNC('month', CURRENT_DATE)
),
not_fix_scheduled AS (
    SELECT hc.*
    FROM household_cohort hc
    LEFT JOIN curated.client_first_conversion cfc
        ON cfc.client_id = hc.client_id
       AND cfc.cancellation_adjusted_first_fix_demand_ts <= hc.onboarding_ts
    WHERE cfc.client_id IS NULL
),
primary_profile_ready AS (
    SELECT nfs.*
    FROM not_fix_scheduled nfs
    JOIN client_service_production.clients csp
        ON csp.client_id = nfs.household_primary_client_id
    WHERE csp.shipping_address IS NOT NULL
      AND EXISTS (
          SELECT 1 FROM payment_method_service.payment_methods pm
          WHERE CAST(pm.client_id AS INTEGER) = nfs.household_primary_client_id
      )
),
not_yet_autoship AS (
    SELECT ppr.*
    FROM primary_profile_ready ppr
    LEFT JOIN curated.client_pulse_journal pj
        ON pj.client_id = ppr.client_id
       AND pj.end_date > DATE_TRUNC('month', CURRENT_DATE) - INTERVAL '1' MONTH
       AND pj.start_date <= DATE(ppr.onboarding_ts)
       AND pj.end_date > DATE(ppr.onboarding_ts)
       AND pj.last_autoship_demand_ts IS NOT NULL
    WHERE pj.client_id IS NULL
)
SELECT
    COUNT(*) AS n_eligible,
    COUNT(DISTINCT DATE(onboarding_ts)) AS days_observed,
    COUNT(*) * 1.0 / COUNT(DISTINCT DATE(onboarding_ts)) AS eligible_per_day
FROM not_yet_autoship
"""

recent_volume_df = query(recent_volume_query)
recent_volume_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_eligible,days_observed,eligible_per_day
0,17284,31,557.5


**Reading this:** August 2026's `eligible_per_day` is 557.5 — identical to May's reference-month figure (verified independently, not a computation error). Volume across March-August 2026 has stayed in a fairly narrow 520-690/day range with no clear downward trend, so the sample-size table in Step 2 uses this figure with reasonable confidence it reflects the population going forward.

In [7]:
DAILY_ELIGIBLE = float(recent_volume_df['eligible_per_day'][0])  # supersedes the Step 1 reference-month figure, per Step 1b
print(f"DAILY_ELIGIBLE (most recent complete month) = {DAILY_ELIGIBLE:,.1f} clients/day")

DAILY_ELIGIBLE (most recent complete month) = 557.5 clients/day


## Step 2 — Sample size & duration, one-sided and two-sided

`n_total_statsmodels` sizes a single pairwise 50/50 comparison; `n_treatment` is read as the **per-arm** requirement, `n_total = 2 x n_per_arm`, and each arm accrues `DAILY_ELIGIBLE / 2` eligible clients per day under the 50/50 split. Both sidedness options are shown side by side — the one-sided table matches the directional hypothesis (Autoship Adoption Rate increases); the two-sided table is the more conservative alternative.

In [8]:
def size_table(rel_grid, baseline, daily, two_sided):
    raw = n_total_statsmodels(
        baseline_rate=baseline, mde_relative=rel_grid, split_ratio=[SPLIT],
        alpha=ALPHA, power=POWER, two_sided=two_sided,
    )
    df = pd.DataFrame(raw).T.reset_index(drop=True)
    df['rel_effect'] = df['mde_relative'].apply(lambda x: f"{x:+.0%}")
    df['n_per_arm'] = df['n_treatment'].astype(int)
    df['n_total'] = df['n_per_arm'] * N_ARMS
    df['days_required'] = np.ceil(df['n_per_arm'] / (daily / N_ARMS)).astype(int)
    df['weeks_required'] = (df['days_required'] / 7).round(1)
    return df[['rel_effect', 'p_treatment', 'n_per_arm', 'n_total', 'days_required', 'weeks_required']]

print(f"--- Autoship Adoption Rate, Treatment vs. Control (baseline={BASELINE_RATE:.1%}, alpha={ALPHA}, power={POWER:.0%}, one-sided, 50/50 split) ---")
one_sided_table = size_table(MDE_GRID, BASELINE_RATE, DAILY_ELIGIBLE, two_sided=False)
one_sided_table

--- Autoship Adoption Rate, Treatment vs. Control (baseline=9.3%, alpha=0.05, power=80%, one-sided, 50/50 split) ---


,rel_effect,p_treatment,n_per_arm,n_total,days_required,weeks_required
0,+2%,0.095367,302401,604802,1085,155.0
1,+3%,0.096302,134996,269992,485,69.3
2,+4%,0.097237,76269,152538,274,39.1
3,+5%,0.098172,49026,98052,176,25.1
4,+10%,0.102847,12523,25046,45,6.4


In [9]:
print(f"--- Autoship Adoption Rate, Treatment vs. Control (baseline={BASELINE_RATE:.1%}, alpha={ALPHA}, power={POWER:.0%}, two-sided, 50/50 split) ---")
two_sided_table = size_table(MDE_GRID, BASELINE_RATE, DAILY_ELIGIBLE, two_sided=True)
two_sided_table

--- Autoship Adoption Rate, Treatment vs. Control (baseline=9.3%, alpha=0.05, power=80%, two-sided, 50/50 split) ---


,rel_effect,p_treatment,n_per_arm,n_total,days_required,weeks_required
0,+2%,0.095367,383903,767806,1378,196.9
1,+3%,0.096302,171379,342758,615,87.9
2,+4%,0.097237,96825,193650,348,49.7
3,+5%,0.098172,62240,124480,224,32.0
4,+10%,0.102847,15898,31796,58,8.3


**Reading this:** the two-sided table always requires more samples (and more days) than the one-sided table for the same MDE and power, since it splits `alpha` across both tails — roughly 25-30% more days at every MDE here. At the observed 557.5/day: the 2% scenario needs 1,085 days one-sided (1,378 two-sided) — well over a year, likely impractical for a monitor-style rollout. 3% needs 485/615 days. 4% needs 274/348 days (~9-11 months). 5% needs 176/224 days. 10% needs just 45/58 days.

## Step 3 — Headline sample-size summary

Headline MDE below is the **median of the Step 2 grid, +4% relative lift**. This summary uses one-sided, matching the directional hypothesis — the two-sided cost at the same MDE is in Step 2's grid, for comparison.

In [10]:
TARGET_REL_MDE = 0.04  # median of the Step 2 grid [0.02, 0.03, 0.04, 0.05, 0.10]

res = n_total_statsmodels(
    baseline_rate=BASELINE_RATE,
    mde_relative=[TARGET_REL_MDE],
    split_ratio=[SPLIT],
    alpha=ALPHA,
    power=POWER,
    two_sided=False,
)

n_per_arm = int(list(res.values())[0]['n_treatment'])
duration_days = int(np.ceil(n_per_arm / (DAILY_ELIGIBLE / N_ARMS)))

summary = {
    'Metric Used': 'Autoship Adoption Rate (Treatment vs. Control)',
    'Population': 'Household Adult + Kids onboarding clients, not already enrolled in Autoship, meeting all 4 eligibility conditions (see Step 0)',
    'Baseline Value': f"{BASELINE_RATE:.1%} (current, un-nudged Control experience; pooled {COHORT_START} through a 90-day-mature cutoff)",
    'Daily Eligible Volume': f"{DAILY_ELIGIBLE:,.1f} / day (most recent complete month, per Step 1b)",
    'Minimum Detectable Effect': f"+{TARGET_REL_MDE:.0%} relative ({BASELINE_RATE:.3f} -> {BASELINE_RATE*(1+TARGET_REL_MDE):.3f})",
    'One/Two-Sided Test': 'One-sided',
    'Significance Level': f"{ALPHA} (single comparison, no multiple-comparison correction)",
    'Statistical Power': f"{POWER:.0%}",
    'Variant Split %': '50% / 50% (Control / Treatment)',
    'Minimum Samples by Variant': f"{n_per_arm:,}",
    'Minimum Samples total': f"{n_per_arm*N_ARMS:,}",
    'Shortest Duration Required': f"{duration_days} days (~{duration_days/7:.1f} weeks)",
}
pd.Series(summary).to_frame('value')

,value
Metric Used,Autoship Adoption Rate (Treatment vs. Control)
Population,"Household Adult + Kids onboarding clients, not already enrolled in Autoship, meeting all 4 eligibility conditions (see Step 0)"
Baseline Value,"9.3% (current, un-nudged Control experience; pooled 2026-03-01 through a 90-day-mature cutoff)"
Daily Eligible Volume,"557.5 / day (most recent complete month, per Step 1b)"
Minimum Detectable Effect,+4% relative (0.093 -> 0.097)
One/Two-Sided Test,One-sided
Significance Level,"0.05 (single comparison, no multiple-comparison correction)"
Statistical Power,80%
Variant Split %,50% / 50% (Control / Treatment)
Minimum Samples by Variant,"76,269"


## Bottom line

- **`MATURATION_DAYS = 90` likely understates the true adoption rate** — Step 0b shows only 66% of eventual (365-day) adopters have shown up by day 90, and the curve is still climbing at day 180 (81%). Extending the checkpoint grid past 180 days is needed before finalizing this number; until then, treat `BASELINE_RATE` (9.35%) as a lower bound, not the final answer.
- **Baseline is the current, un-nudged adoption rate** — Household Adult + Kids clients see no Autoship nudge today, so this measures what Control looks like, not a proxy borrowed from a different population.
- **No live allocation log exists yet** — this experiment has not launched, so both the rate and the volume come from historical eligibility reads rather than an in-flight allocation log; refresh both once the experiment is live and an allocation log exists.
- **Volume has been stable, not declining** — 520-690 eligible/day from March through August 2026, with no clear downward trend across that window.
- At the **+4% relative** headline MDE, one-sided: 76,269 per arm, 274 days (~39 weeks). The full grid across 2-10%, both sidedness options, is in Step 2 — note the 2-3% scenarios run well over a year and are likely impractical.